In [1]:
k = 0.0008

# --- Delivery Time Opportunity Cost (k_s) ---
# Unit: 
#     Proportion of SKU price per HOUR of delivery time per UNIT shipped.
# Interpretation: 
#     Each additional delivery hour costs 0.08% of SKU price per unit shipped.

# Industry reference:
#     E-commerce research indicates that a one-day delivery delay can reduce conversion rates by approximately 1%–3% (e.g., studies on Amazon Prime logistics impact).
# We conservatively assume:
#     1 day delay ≈ 2% loss in revenue opportunity.
#     Since delivery time in our dataset is measured in HOURS, 2% per day → 0.02 / 24 ≈ 0.0008 per hour.

# Modeling implication: 
#     Encourages shorter delivery routes but does not completely dominate other costs.
# Sensitivity analysis suggestion:
#     Test k in {0.0004, 0.00125}.
#     - Lower k (0.0004): model tolerates longer shipping times.
#     - Higher k (0.00125): model strongly prioritizes faster routes and local fulfillment.


In [2]:
h = 0.00055

# --- Holding Cost (h_s) ---
# Unit: 
#     Proportion of SKU price per DAY.
# Interpretation:
#     Each unit of inventory incurs 0.055% of its price as holding cost per day.

# Industry reference: 
#     Inventory holding cost is typically 15%–30% of product value per YEAR.
#     (Chopra & Meindl, "Designing and Managing the Supply Chain"; Simchi-Levi et al., "Supply Chain Management").
# We assume: 
#     a 20% annual holding cost rate, which is a common midpoint in retail practice.
#     Since our model operates on a daily time scale, we convert annual rate to daily: h = 0.20 / 365 ≈ 0.00055 per day.

# Modeling implication:
#     This value moderately penalizes excess inventory without forcing inventory to zero.
# Sensitivity analysis suggestion:
#     Test h in {0.0004, 0.0008}.
#     - Lower h (0.0004): encourages higher inventory levels.
#     - Higher h (0.0008): strongly discourages inventory holding.

In [3]:
u = 0.07

# --- Purchasing Opportunity Cost (u_s) ---
# Unit: 
#     Proportion of SKU price per UNIT purchased
# Interpretation:
#     Each unit purchased incurs an additional 7% of SKU price as opportunity cost.

# Industry reference:
#     Emergency replenishment or small-batch procurement often involves a 3%–10% premium due to reduced bargaining power, expedited logistics, or supplier flexibility costs.
#     (Supported by retail operations reports from McKinsey and BCG.)
# We assume: 
#     a 7% purchasing premium.

# Modeling implication:
#     This discourages excessive procurement and encourages internal redistribution when possible.
# Sensitivity analysis suggestion:
#     Test u in {0.03, 0.10}
#     - Lower u (0.03): model favors procurement over redistribution.
#     - Higher u (0.10): model strongly avoids purchasing and prefers inter-warehouse transfer.

In [4]:
# --- price of the clusters (p_s) ---
# we use cluster price to replace sku price. 
# As long as we obtain the information 'which skus are in one specific cluster', we can calculate the price of the cluster by simply adding the price of skus it contains.

import pandas as pd
import numpy as np

# File 1: sku_cluster_data.xlsx
# This file defines which SKU belongs to which cluster.
# Expected key columns:
#   - cluster (or cluster_id): cluster identifier
#   - sku_ID: SKU identifier
#sku_cluster_data.xlsx
sku_cluster_df = pd.read_csv("../data/processed/sku_warehouse_train_test_clusters_assignments.csv")

# File 2: ../data/raw/JD_order_data.csv
# This file is used to derive SKU-level price.
# Expected key columns:
#   - sku_ID
#   - original_unit_price
#   - quantity

order_df = pd.read_csv("../data/raw/JD_order_data.csv")

# Clean column names
sku_cluster_df.columns = sku_cluster_df.columns.str.strip()
order_df.columns = order_df.columns.str.strip()


cluster_col = "sku_cluster_ID"
sku_col = "sku_ID"
price_col = "original_unit_price"
qty_col = "quantity"


# ============================================
# Step 1: Compute row-level unit price in JD_order_data
# ============================================

# Modeling logic:
# For each row in JD_order_data, the SKU unit price is defined as:
#     original_unit_price / quantity

# We exclude rows with missing or non-positive quantity to avoid invalid division.

order_price_df = order_df[[sku_col, price_col, qty_col]].copy()
order_price_df = order_price_df.dropna(subset=[sku_col, price_col, qty_col])
order_price_df = order_price_df[order_price_df[qty_col] > 0].copy()

order_price_df["sku_unit_price"] = order_price_df[price_col]
#/ order_price_df[qty_col]

# ============================================
# Step 2: Compute final price for each SKU
# ============================================

# For each sku_ID:
#   1. find all rows with that sku_ID
#   2. compute row-level unit price = original_unit_price / quantity
#   3. average these row-level unit prices
#
# Output:
#   sku_price_df with columns:
#       sku_ID, sku_price

sku_price_df = (
    order_price_df
    .groupby(sku_col, as_index=False)["sku_unit_price"]
    .mean()
    .rename(columns={"sku_unit_price": "sku_price"})
)

# ============================================
# Step 3: Merge SKU prices into cluster mapping
# ============================================

# This attaches each SKU's final price to its cluster.

cluster_price_input_df = sku_cluster_df.merge(
    sku_price_df,
    on=sku_col,
    how="left"
)

# ============================================
# Step 4: Compute cluster-level price p_s
# ============================================

# Modeling logic:
# Each cluster contains multiple SKUs.
# The cluster price p_s is defined as the average SKU price
# across all SKUs belonging to that cluster.
#
# Output:
#   p[cluster] = average sku_price within that cluster


cluster_price_df = (
    cluster_price_input_df
    .groupby(cluster_col, as_index=False)["sku_price"]
    .mean()
    .rename(columns={"sku_price": "p_s"})
)

# ============================================
# Step 5: Convert to dictionary for optimization model
# ============================================

# Final parameter:
#   p[s] where s is cluster identifier

p = dict(zip(cluster_price_df[cluster_col], cluster_price_df["p_s"]))

# ============================================
# Step 6: Optional diagnostics
# ============================================

print("Number of SKUs with computed prices:", sku_price_df[sku_col].nunique())
print("Number of clusters with computed prices:", cluster_price_df[cluster_col].nunique())

missing_price_rows = cluster_price_input_df["sku_price"].isna().sum()
print("Number of SKU-cluster rows with missing SKU price:", missing_price_rows)

# Show SKU-level prices
print("\nSKU-level prices:")
display(sku_price_df.head())

# Show cluster-level prices
print("\nCluster-level prices p_s:")
display(cluster_price_df.head())

# Show dictionary
print("\nParameter p_s as dictionary:")
for k, v in list(p.items())[:10]:
    print(f"{k}: {v}")


Number of SKUs with computed prices: 9159
Number of clusters with computed prices: 22
Number of SKU-cluster rows with missing SKU price: 30

SKU-level prices:


,sku_ID,sku_price
0,000aa92b82,83.333333
1,000d4af39d,159.000000
2,000dc27e13,116.000000
3,000e84e3a7,75.000000
4,00104dbcd7,79.000000



Cluster-level prices p_s:


,sku_cluster_ID,p_s
0,0,87.140313
1,1,135.092091
2,2,39.691183
3,3,101.303772
4,4,109.319696



Parameter p_s as dictionary:
0: 87.14031269785139
1: 135.09209090948184
2: 39.69118349349238
3: 101.30377224959882
4: 109.31969608778552
5: 10581.4
6: 305.224141238395
7: 116.55612580647721
8: 148.57767343056437
9: 219.40553910683445


In [5]:
import pandas as pd
import numpy as np

# Data source:
#     The file "DCs cross-filling.csv" contains inter-warehouse routing information.
# Relevant columns:
#     - dc_ori: origin distribution center (warehouse n)
#     - dc_des: destination distribution center (warehouse m)
#     - cross_filling_hours: delivery time (in hours) from n to m

cross_fill_df = pd.read_csv("../data/processed/JD_sku_dc_cross_filling_hours.csv")

# Ensure consistent column naming (remove extra spaces if necessary)
cross_fill_df.columns = cross_fill_df.columns.str.strip()



# --- Construct Delivery Time Matrix t_{n,m} ---

# Unit:
#      cross_filling_hours is measured in HOURS per shipment route. Therefore, t_{n,m} represents delivery time in hours per unit shipped.


t_matrix = cross_fill_df.pivot_table(
    index="dc_ori",
    columns="dc_des",
    values="cross_filling_hours",
    aggfunc="mean"
)



# --- Construct Route Availability Matrix r_{n,m} ---

# Definition:
#      r_{n,m} = 1  if a route from warehouse n to warehouse m exists in the dataset
#      r_{n,m} = 0  otherwise
#      The existence of a (dc_ori, dc_des) pair in "DCs cross-filling.csv" indicates that operational cross-filling is feasible between the two DCs.


# Create binary indicator for route existence
cross_fill_df["route_exists"] = 1

r_matrix = cross_fill_df.pivot_table(
    index="dc_ori",
    columns="dc_des",
    values="route_exists",
    aggfunc="max"
)

# Replace NaN with 0 (no route exists)
r_matrix = r_matrix.fillna(0)

# Ensure binary integer type
r_matrix = r_matrix.astype(int)


# Ensure both matrices share identical index and column sets
all_dcs = sorted(set(cross_fill_df["dc_ori"]).union(set(cross_fill_df["dc_des"])))

t_matrix = t_matrix.reindex(index=all_dcs, columns=all_dcs)
r_matrix = r_matrix.reindex(index=all_dcs, columns=all_dcs).fillna(0).astype(int)


# Final Output Objects

print("Delivery Time Matrix (t_nm):")
display(t_matrix)

print("Route Availability Matrix (r_nm):")
display(r_matrix)

Delivery Time Matrix (t_nm):


dc_des,1,2,3,4,5,6,7,8,9,10,...,58,59,60,61,62,63,64,65,66,67
dc_ori,,,,,,,,,,,,,,,,,,,,,
1,13.674569,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,13.348242,NaN,NaN,NaN,24.209856,47.859155,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,18.938967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,17.540891,NaN,NaN,NaN
4,NaN,NaN,NaN,20.054066,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,54.657431,NaN,13.617163,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,47.420712,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,NaN,58.863636,NaN,31.508571,NaN,57.483333,48.800000,NaN,44.000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
64,NaN,NaN,44.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,12.771889,NaN,NaN,NaN
65,83.000000,60.888889,39.166667,77.272727,16.809524,59.083333,59.411765,66.0,73.925,49.191176,...,NaN,NaN,83.5,92.5,NaN,NaN,33.200000,NaN,NaN,NaN


Route Availability Matrix (r_nm):


dc_des,1,2,3,4,5,6,7,8,9,10,...,58,59,60,61,62,63,64,65,66,67
dc_ori,,,,,,,,,,,,,,,,,,,,,
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,0,1,0,1,0,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
64,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
65,1,1,1,1,1,1,1,1,1,1,...,0,0,1,1,0,0,1,0,0,0


In [6]:
import pandas as pd

# Data source:
#     The file "Capacity.csv" contains warehouse capacity information.
# Relevant columns:
#     - dc_des: warehouse identifier (j)
#     - capacity: total storage capacity of warehouse j

# Interpretation:
#     C_j represents the maximum total inventory (sum over all SKUs) that warehouse j can hold.


capacity_df = pd.read_csv("../data/processed/inventory_capacity.csv")

# Clean column names (remove possible spaces)
capacity_df.columns = capacity_df.columns.str.strip()



# --- Capacity Parameter C_j ---

C = dict(zip(capacity_df["dc_des"], capacity_df["capacity"]))

# Optional: Ensure capacity is numeric

C = {j: float(C[j]) for j in C}

# Display result

print("Warehouse Capacity Parameter C_j:")
for j in C:
    print(f"Warehouse {j}: Capacity = {C[j]}")

Warehouse Capacity Parameter C_j:
Warehouse 1: Capacity = 205.0
Warehouse 2: Capacity = 2949.0
Warehouse 3: Capacity = 192.0
Warehouse 4: Capacity = 3443.0
Warehouse 5: Capacity = 3805.0
Warehouse 6: Capacity = 810.0
Warehouse 7: Capacity = 1130.0
Warehouse 8: Capacity = 103.0
Warehouse 9: Capacity = 3215.0
Warehouse 10: Capacity = 851.0
Warehouse 11: Capacity = 96.0
Warehouse 12: Capacity = 48.0
Warehouse 13: Capacity = 101.0
Warehouse 14: Capacity = 78.0
Warehouse 15: Capacity = 149.0
Warehouse 16: Capacity = 885.0
Warehouse 17: Capacity = 1016.0
Warehouse 18: Capacity = 130.0
Warehouse 19: Capacity = 125.0
Warehouse 20: Capacity = 1201.0
Warehouse 21: Capacity = 64.0
Warehouse 22: Capacity = 83.0
Warehouse 23: Capacity = 45.0
Warehouse 24: Capacity = 1511.0
Warehouse 25: Capacity = 270.0
Warehouse 26: Capacity = 922.0
Warehouse 27: Capacity = 1807.0
Warehouse 28: Capacity = 937.0
Warehouse 29: Capacity = 86.0
Warehouse 30: Capacity = 173.0
Warehouse 31: Capacity = 1006.0
Warehouse 3

In [7]:
# --- the demand of each cluster in j warehouse (D_sj) ---
# obtain from the customer side

import pandas as pd
import numpy as np

# Data source:
#   - training_daily_demand.xlsx covers 2024-03-01 to 2024-03-24
#   - testing_daily_demand.xlsx covers 2024-03-25 to 2024-03-31
#
# Expected key columns:
#   - order_date : date of demand observation
#   - dc_des     : destination warehouse j
#   - cluster    : demand cluster s
#   - demand     : daily demand quantity for cluster s at warehouse j on that date
#
# Final goal:
#   Compute D_{s,j}, defined as the average daily demand over the full 31-day period
#   for each cluster s and warehouse j.


# testing_daily_demand.xlsx; training_daily_demand.xlsx

train_df = pd.read_csv("../data/processed/sku_warehouse_train_test_clusters_train_warehouse_daily_demand.csv")
test_df = pd.read_csv("../data/processed/sku_warehouse_train_test_clusters_test_warehouse_daily_demand.csv")

# Clean column names
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

# ============================================
# Define column names
# ============================================

date_col = "order_date"
warehouse_col = "dc_des"
cluster_col = "sku_cluster_ID"
demand_col = "demand"

# ============================================
# Combine training and testing data
# ============================================

demand_df = pd.concat([train_df, test_df], ignore_index=True)

# Convert order_date to datetime
demand_df[date_col] = pd.to_datetime(demand_df[date_col])

# Keep only the target 31-day window if needed
demand_df = demand_df[
    (demand_df[date_col] >= "2018-03-01") &
    (demand_df[date_col] <= "2018-03-31")
].copy()

# ============================================
# Aggregate to daily warehouse-cluster demand
# ============================================

# Rationale:
# If there are multiple rows for the same warehouse, cluster, and date, they should be summed to obtain the total demand of that cluster at that warehouse on that day.

daily_demand_df = (
    demand_df
    .groupby([date_col, warehouse_col, cluster_col], as_index=False)[demand_col]
    .sum()
)

# ============================================
# Create complete 31-day panel for each (warehouse, cluster)
# ============================================

# Modeling choice:
# D_{s,j} is defined as the average daily demand over the entire 31-day period.
# Therefore, if a warehouse-cluster pair has no record on some dates, we treat missing daily demand as 0 rather than dropping those days.

all_dates = pd.date_range(start="2018-03-01", end="2018-03-31", freq="D")

all_warehouses = sorted(daily_demand_df[warehouse_col].dropna().unique())
all_clusters = sorted(daily_demand_df[cluster_col].dropna().unique())

full_index = pd.MultiIndex.from_product(
    [all_dates, all_warehouses, all_clusters],
    names=[date_col, warehouse_col, cluster_col]
)

daily_demand_full_df = (
    daily_demand_df
    .set_index([date_col, warehouse_col, cluster_col])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# ============================================
# Compute D_{s,j}: average daily demand over 31 days
# ============================================

D_df = (
    daily_demand_full_df
    .groupby([cluster_col, warehouse_col], as_index=False)[demand_col]
    .mean()
    .rename(columns={demand_col: "D_sj"})
)

# ============================================
# Convert to dictionary for optimization model
# ============================================

# Final parameter:
#   D[(s, j)] = average daily demand of cluster s at warehouse j

D = dict(zip(zip(D_df[cluster_col], D_df[warehouse_col]), D_df["D_sj"]))

# ============================================
# Optional diagnostics
# ============================================

print("Number of warehouse-cluster demand pairs:", len(D_df))
print("Date range used:", daily_demand_full_df[date_col].min(), "to", daily_demand_full_df[date_col].max())
print("Number of days included:", daily_demand_full_df[date_col].nunique())

print("\nSample of D_sj table:")
display(D_df.head())

print("\nSample of D[(s,j)] dictionary:")
for k, v in list(D.items())[:10]:
    print(f"{k}: {v}")

Number of warehouse-cluster demand pairs: 1320
Date range used: 2018-03-01 00:00:00 to 2018-03-31 00:00:00
Number of days included: 31

Sample of D_sj table:


,sku_cluster_ID,dc_des,D_sj
0,0,1,11.064516
1,0,2,121.903226
2,0,3,12.516129
3,0,4,127.645161
4,0,5,138.387097



Sample of D[(s,j)] dictionary:
(0, 1): 11.064516129032258
(0, 2): 121.90322580645162
(0, 3): 12.516129032258064
(0, 4): 127.64516129032258
(0, 5): 138.38709677419354
(0, 6): 44.193548387096776
(0, 7): 49.25806451612903
(0, 8): 5.580645161290323
(0, 9): 150.58064516129033
(0, 10): 34.16129032258065


In [8]:
# --- Whether the warehouse is designated and able to purchase (W_j) ---

import pandas as pd
import numpy as np

# ============================================
# Load network data
# ============================================

# Data source:
# ../data/raw/JD_network_data.csv contains warehouse network information.
# Relevant columns:
#   - dc_ID: warehouse identifier j
#   - region_ID: regional warehouse identifier
#
# Modeling logic:
# A warehouse j is considered a large warehouse if dc_ID == region_ID.
# Large warehouses are assumed to have procurement authority, so W_j = 1.
# Otherwise, the warehouse is a small warehouse without procurement authority, so W_j = 0.

network_df = pd.read_csv("../data/raw/JD_network_data.csv")

# Clean column names
network_df.columns = network_df.columns.str.strip()

# ============================================
# Define column names
# ============================================

warehouse_col = "dc_ID"
region_col = "region_ID"

# ============================================
# Construct procurement eligibility parameter W_j
# ============================================

# W_j is a binary parameter:
#   W_j = 1 if warehouse j is a large warehouse (dc_ID == region_ID)
#   W_j = 0 otherwise

network_df["W_j"] = (network_df[warehouse_col] == network_df[region_col]).astype(int)

# ============================================
# Keep one record per warehouse
# ============================================

# If the file contains repeated rows for the same dc_ID,
# use max() so that a warehouse is marked as eligible for procurement if any row indicates it is a large warehouse.

W_df = (
    network_df
    .groupby(warehouse_col, as_index=False)["W_j"]
    .max()
    .rename(columns={warehouse_col: "warehouse"})
)

# ============================================
# Convert to dictionary for optimization model
# ============================================

# Final parameter:
#   W[j] = 1 if warehouse j can purchase
#   W[j] = 0 otherwise

W = dict(zip(W_df["warehouse"], W_df["W_j"]))

# ============================================
# Diagnostics
# ============================================

print("Number of warehouses:", len(W_df))
print("Number of large warehouses (W_j = 1):", int(W_df["W_j"].sum()))
print("Number of small warehouses (W_j = 0):", int((W_df["W_j"] == 0).sum()))

print("\nSample of W_j table:")
display(W_df.head())

print("\nSample of W[j] dictionary:")
for k, v in list(W.items())[:10]:
    print(f"{k}: {v}")

Number of warehouses: 56
Number of large warehouses (W_j = 1): 8
Number of small warehouses (W_j = 0): 48

Sample of W_j table:


,warehouse,W_j
0,1,0
1,2,1
2,3,1
3,4,1
4,5,1



Sample of W[j] dictionary:
1: 0
2: 1
3: 1
4: 1
5: 1
6: 0
7: 1
8: 0
9: 1
10: 1


In [9]:
# --- inventory level at the beginning of the date (I0_sj) ---

import pandas as pd
import numpy as np

# ============================================
# Load data
# ============================================

# ../data/raw/JD_inventory_data.csv:
# contains inventory presence records by date, warehouse, and SKU.
# Since exact inventory quantity is unavailable, each observed SKU record is assumed to represent 1 unit of inventory.

inventory_df = pd.read_csv("../data/raw/JD_inventory_data.csv")

# sku_cluster_data.xlsx:
# maps each SKU_ID to its corresponding cluster (sku_cluster_ID)

sku_cluster_df = pd.read_csv("../data/processed/sku_warehouse_train_test_clusters_assignments.csv")

# Clean column names
inventory_df.columns = inventory_df.columns.str.strip()
sku_cluster_df.columns = sku_cluster_df.columns.str.strip()

# ============================================
# Define column names
# ============================================

date_col = "date"
warehouse_col = "dc_ID"
sku_col = "sku_ID"
cluster_col = "sku_cluster_ID"

# ============================================
# Convert date column to datetime
# ============================================

inventory_df[date_col] = pd.to_datetime(inventory_df[date_col])

# Keep only the 31-day period used in the model
inventory_df = inventory_df[
    (inventory_df[date_col] >= "2018-03-01") &
    (inventory_df[date_col] <= "2018-03-31")
].copy()

# ============================================
# Merge inventory records with cluster mapping
# ============================================

inv_cluster_df = inventory_df.merge(
    sku_cluster_df[[sku_col, cluster_col]],
    on=sku_col,
    how="left"
)

# Drop rows where cluster mapping is missing
inv_cluster_df = inv_cluster_df.dropna(subset=[cluster_col]).copy()

# ============================================
# Inventory quantity assumption
# ============================================

# Since JD_inventory_data does not provide actual inventory quantity for each SKU, we assume that each observed inventory record corresponds to 1 unit of inventory.
# Therefore, each row contributes 1 unit to the cluster-level daily inventory.

inv_cluster_df["inventory_unit"] = 1

# ============================================
# Compute daily cluster inventory at each warehouse
# ============================================

# For each date, warehouse, and cluster:
# cluster daily inventory = sum of inventory_unit across all SKU records

daily_inventory_df = (
    inv_cluster_df
    .groupby([date_col, warehouse_col, cluster_col], as_index=False)["inventory_unit"]
    .sum()
    .rename(columns={"inventory_unit": "daily_cluster_inventory"})
)

# ============================================
# Build complete 31-day panel
# ============================================

# Missing dates are filled with 0 inventory.
# This ensures that I0_sj is averaged over the full 31-day window.

all_dates = pd.date_range(start="2018-03-01", end="2018-03-31", freq="D")
all_warehouses = sorted(daily_inventory_df[warehouse_col].dropna().unique())
all_clusters = sorted(daily_inventory_df[cluster_col].dropna().unique())

full_index = pd.MultiIndex.from_product(
    [all_dates, all_warehouses, all_clusters],
    names=[date_col, warehouse_col, cluster_col]
)

daily_inventory_full_df = (
    daily_inventory_df
    .set_index([date_col, warehouse_col, cluster_col])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# ============================================
# Compute I0_{s,j}
# ============================================

# I0_sj is defined as the average daily inventory of cluster s
# at warehouse j over the 31-day period.

I0_df = (
    daily_inventory_full_df
    .groupby([cluster_col, warehouse_col], as_index=False)["daily_cluster_inventory"]
    .mean()
    .rename(columns={"daily_cluster_inventory": "I0_sj"})
)

# ============================================
# Convert to dictionary for optimization model
# ============================================

# Final parameter:
#   I0[(s, j)] = average daily initial inventory of cluster s at warehouse j

I0 = dict(zip(zip(I0_df[cluster_col], I0_df[warehouse_col]), I0_df["I0_sj"]))

# ============================================
# Diagnostics
# ============================================

print("Number of warehouse-cluster inventory pairs:", len(I0_df))
print("Date range used:", daily_inventory_full_df[date_col].min(), "to", daily_inventory_full_df[date_col].max())
print("Number of days included:", daily_inventory_full_df[date_col].nunique())

print("\nSample of daily cluster inventory:")
display(daily_inventory_df.head())

print("\nSample of I0_sj table:")
display(I0_df.head())

print("\nSample of I0[(s,j)] dictionary:")
for k, v in list(I0.items())[:10]:
    print(f"{k}: {v}")

Number of warehouse-cluster inventory pairs: 1120
Date range used: 2018-03-01 00:00:00 to 2018-03-31 00:00:00
Number of days included: 31

Sample of daily cluster inventory:


,date,dc_ID,sku_cluster_ID,daily_cluster_inventory
0,2018-03-01,1,2.0,1
1,2018-03-01,1,4.0,7
2,2018-03-01,1,6.0,1
3,2018-03-01,1,11.0,1
4,2018-03-01,1,12.0,1



Sample of I0_sj table:


,sku_cluster_ID,dc_ID,I0_sj
0,0.0,1,0.354839
1,0.0,2,24.354839
2,0.0,3,20.645161
3,0.0,4,18.903226
4,0.0,5,22.548387



Sample of I0[(s,j)] dictionary:
(0.0, 1): 0.3548387096774194
(0.0, 2): 24.35483870967742
(0.0, 3): 20.64516129032258
(0.0, 4): 18.903225806451612
(0.0, 5): 22.548387096774192
(0.0, 6): 0.45161290322580644
(0.0, 7): 19.870967741935484
(0.0, 8): 1.0
(0.0, 9): 22.70967741935484
(0.0, 10): 25.70967741935484
